[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C41_Deep_RL_Course/03_offline_rl/03_offline_rl.ipynb)

# 03 · 离线强化学习（纯 numpy，从零）

目标：在一个**固定数据集**（不许再交互）上，亲眼看到**分布偏移**如何让朴素 Q-learning 崩坏，再实现 **BC**、**CQL 保守惩罚**、**IQL expectile 回归** 三种应对，对比它们。

路线：建固定数据集 + 覆盖分析 → 朴素 FQI 的 OOD 高估崩坏 → BC → CQL 保守惩罚 → IQL expectile → ✏️ 练习(BC/分布偏移度量/CQL惩罚/expectile) → 📖 答案 → 🧪 真实基准胶囊。

> 心智模型：**离线的死结 = OOD 动作被 max 高估却永不被纠正；BC 不偏离、CQL 压低 OOD、IQL 不查 OOD**。

## 1 · 固定数据集与覆盖分析

用一个 1D 链 MDP（状态 0..N-1，动作 左/右，到 N-1 得 +1）。
**关键**：行为策略在状态 2 **从不向左**，所以 `(s=2, 左)` 是 OOD（数据未覆盖）——离线 RL 的危险点。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
N, nA, gamma = 6, 2, 0.9       # 动作 0=左 1=右

def step(s, a):
    s2 = min(s+1, N-1) if a == 1 else max(s-1, 0)
    done = (s2 == N-1)
    return s2, (1.0 if done else 0.0), done

def collect_dataset(n_ep=300, seed=0):
    r = np.random.default_rng(seed); data = []
    for _ in range(n_ep):
        s = int(r.integers(0, N-1))
        for _ in range(20):
            if s == 2:
                a = 1                      # 行为策略在 s=2 从不向左 -> (2,左) 是 OOD
            else:
                a = 1 if r.random() < 0.9 else 0
            s2, rew, done = step(s, a)
            data.append((s, a, rew, s2, float(done))); s = s2
            if done: break
    return np.array(data)

data = collect_dataset()
cov = np.zeros((N, nA))
for s, a, *_ in data: cov[int(s), int(a)] += 1
print(f'数据集 {len(data)} 条转移，(s,a) 覆盖计数:')
print(cov.astype(int))
assert cov[2, 0] == 0, '(s=2, 左) 应未被覆盖(OOD)'
print('\n关键：(s=2, 左) 计数 = 0 -> 它是 OOD 动作，Q 网络只能外推、无真实回报纠正')
print('✅ 固定数据集就绪，OOD 点已埋好')

## 2 · 朴素 FQI：OOD 高估永不被纠正

标准离线 Q-iteration：`Q(s,a) ← r + γ max_a' Q(s',a')`。

**模拟函数逼近的外推**：给 OOD 动作 `(2,左)` 一个虚高初值 5.0（真实网络会这样外推）。看朴素 FQI **不但不修正它，还让它顶高周围所有状态的 Q**。

In [ ]:
def naive_fqi(data, ood_init=5.0, iters=200, lr=0.05):
    Q = np.zeros((N, nA))
    Q[2, 0] = ood_init                 # 模拟网络对 OOD 动作的虚高外推
    for _ in range(iters):
        Qn = Q.copy()
        for s, a, r, s2, d in data:
            s, a, s2 = int(s), int(a), int(s2)
            y = r + gamma * (1 - d) * Q[s2].max()    # max 会追逐 OOD 高估
            Qn[s, a] += lr * (y - Qn[s, a])
        Q = Qn
    return Q

Q_naive = naive_fqi(data, ood_init=5.0)
print('朴素 FQI 学到的 Q（OOD 初值 5.0）:')
print(np.round(Q_naive, 2))
print(f'\nOOD 动作 Q[2,左] = {Q_naive[2,0]:.2f} (从未被数据更新 -> 虚高保留)')
# 真实最优 Q 全部 <= 1（最大回报是 1）。OOD 高估让很多 Q 远超 1
n_inflated = int((Q_naive > 1.01).sum())
print(f'被高估到 >1 的 (s,a) 数 = {n_inflated} (真实最优 Q 都应 <= 1!)')
assert Q_naive[2, 0] > 4.0, 'OOD 动作 Q 应保持虚高(未被纠正)'
assert n_inflated >= 3, 'OOD 高估应经 max 自举污染多个状态'
print('💥 离线死结：OOD 高估永不被现实纠正，还经 max 自举污染全局')

## 3 · 行为克隆（BC）：安全但有天花板

BC 把数据当监督学习，统计每个状态下各动作的频率作为策略（表格 BC = 经验动作分布）。
它**只模仿数据、不偏离**，所以没有 OOD 问题——但学不会超过数据。

In [ ]:
def behavior_cloning(data):
    '''表格 BC：π(a|s) = 数据中 (s,a) 频率。'''
    counts = np.zeros((N, nA))
    for s, a, *_ in data: counts[int(s), int(a)] += 1
    pi = np.zeros((N, nA))
    for s in range(N):
        tot = counts[s].sum()
        pi[s] = counts[s] / tot if tot > 0 else np.ones(nA) / nA
    return pi

pi_bc = behavior_cloning(data)
print('BC 策略 π(a|s) [左, 右]:')
for s in range(N): print(f'  s={s}: {np.round(pi_bc[s], 2)}')
# BC 在 s=2 只会向右(数据如此)——恰好也对，但它纯靠模仿，不理解为什么
assert pi_bc[2, 0] == 0.0, 'BC 复刻数据：s=2 从不向左'
assert pi_bc[2, 1] == 1.0, 'BC 在 s=2 总向右(模仿)'
# BC 永不输出 OOD 动作 -> 安全；但水平上限 = 行为策略
greedy_bc = pi_bc.argmax(1)
print('BC 贪婪动作:', greedy_bc, '(2=右朝目标，安全但=数据水平)')
print('✅ BC 安全(只模仿数据、永不碰OOD)，但天花板=采数据的策略')

## 4 · CQL：保守压低 OOD 动作的 Q

在 TD 更新外加**保守项**：压低 `logsumexp_a Q(s,a)`（软 max，主要压被高估的 OOD），抬高数据内动作 Q。
保守项对 Q 的梯度 = `softmax(Q[s]) - onehot(a_data)`。看它把虚高的 OOD Q 压下去。

In [ ]:
def cql(data, alpha=1.0, ood_init=5.0, iters=300, lr=0.05):
    Q = np.zeros((N, nA)); Q[2, 0] = ood_init
    for _ in range(iters):
        Qn = Q.copy()
        for s, a, r, s2, d in data:
            s, a, s2 = int(s), int(a), int(s2)
            y = r + gamma * (1 - d) * Q[s2].max()
            Qn[s, a] += lr * (y - Qn[s, a])            # 标准 TD
            soft = np.exp(Q[s]) / np.exp(Q[s]).sum()   # softmax (logsumexp 的梯度)
            Qn[s] -= lr * alpha * soft                 # 压低所有动作(尤其高的)
            Qn[s, a] += lr * alpha                     # 抬高数据内动作
        Q = Qn
    return Q

Q_cql = cql(data, alpha=1.0, ood_init=5.0)
print('CQL 学到的 Q (alpha=1):')
print(np.round(Q_cql, 2))
print(f'\nOOD 动作 Q[2,左]: 朴素 {Q_naive[2,0]:.2f} -> CQL {Q_cql[2,0]:.2f}')
assert Q_cql[2, 0] < Q_naive[2, 0] - 1.0, 'CQL 应把 OOD 动作 Q 大幅压低'
assert Q_cql[2, 0] < Q_cql[2, 1], 'CQL 后 OOD(左) 的 Q 应低于数据内(右)'
print('✅ CQL 把虚高的 OOD Q 压到数据内动作之下 -> max 不再追逐它')

## 5 · IQL：expectile 回归在数据内近似 max

IQL 不查 OOD：用**高-τ expectile 回归**学 `V(s)` 来近似数据内动作的最大 Q。
expectile = 非对称平方损失，`τ→1` 时解逼近 max、`τ=0.5` 退化为均值。先把 expectile 机制看清。

In [ ]:
def expectile_grad(pred, target, tau):
    '''expectile L2: 权重 tau(高估侧)/1-tau(低估侧)。返回对 pred 的梯度。'''
    diff = target - pred
    w = np.where(diff > 0, tau, 1 - tau)
    return -2 * w * diff

def fit_expectile(values, tau, iters=3000, lr=0.01):
    '''拟合一个标量到一组 values 的 tau-expectile。'''
    v = 0.0
    for _ in range(iters):
        v -= lr * expectile_grad(np.full_like(values, v), values, tau).mean()
    return v

# 一组数(含一个大值)，看 tau 从 0.5 -> 1 如何从均值滑向 max
vals = np.array([0.0, 0.5, 1.0, 2.0, 5.0])
for tau in [0.5, 0.9, 0.99]:
    print(f'tau={tau}: expectile={fit_expectile(vals, tau):.3f}  (均值={vals.mean():.2f}, max={vals.max():.2f})')
v_mean = fit_expectile(vals, 0.5); v_max = fit_expectile(vals, 0.99)
assert abs(v_mean - vals.mean()) < 0.05, 'tau=0.5 -> 均值'
assert v_max > v_mean + 1.0, 'tau->1 -> 偏向 max'
print('✅ expectile: tau=0.5→均值, tau→1→近似max —— IQL 用它在「只看数据内动作」时近似 max，绕开 OOD')

### 把 expectile 用进 IQL 学 V

用高-τ expectile 学 `V(s) ≈ max_{a∈data} Q(s,a)`，全程只用数据内动作。

In [ ]:
def iql_learn_V(data, Q, tau=0.8, iters=2000, lr=0.02):
    '''V(s) <- tau-expectile of Q(s, a_data)，只用数据里出现的 (s,a)。'''
    V = np.zeros(N)
    # 收集每个状态在数据里出现过的动作的 Q 值
    sa_in_data = {s: [] for s in range(N)}
    for s, a, *_ in data:
        sa_in_data[int(s)].append(Q[int(s), int(a)])
    for _ in range(iters):
        for s in range(N):
            if sa_in_data[s]:
                tgt = np.array(sa_in_data[s])
                V[s] -= lr * expectile_grad(np.full_like(tgt, V[s]), tgt, tau).mean()
    return V

# 用一个合理的(数据内)Q：朴素 FQI 但不给 OOD 虚高(ood_init=0)
Q_clean = naive_fqi(data, ood_init=0.0)
V_iql = iql_learn_V(data, Q_clean, tau=0.8)
print('IQL 的 V(s) (高-tau expectile, 只用数据内动作):', np.round(V_iql, 3))
# V 应接近每个状态数据内动作的最大 Q，且不被任何 OOD 动作影响
for s in range(N-1):
    data_actions = [int(a) for ss,a,*_ in data if int(ss)==s]
    if data_actions:
        max_in_data = max(Q_clean[s, a] for a in set(data_actions))
        assert V_iql[s] <= max_in_data + 0.1, 'V 不应超过数据内动作的 max(没碰OOD)'
print('✅ IQL 的 V 在数据内近似 max、完全不查询 OOD 动作 -> 结构性避开分布偏移')

---
## ✏️ 练习 1：行为克隆的贪婪策略

实现 `bc_greedy_policy(data)`：返回长度 N 的数组，每个状态取数据中**最常见**的动作（表格 BC 的贪婪化）。

In [ ]:
def bc_greedy_policy(data):
    # TODO: 统计每个状态各动作计数，返回每状态 argmax 动作(长度 N 的 int 数组)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
data = collect_dataset()
g = bc_greedy_policy(data)
assert len(g) == N
assert g[2] == 1, 's=2 数据里只有右 -> BC 贪婪取右'
assert g[4] == 1, 's=4 数据里多为右 -> 取右'
print('BC 贪婪策略:', g)
print('✅ 练习 1 通过：BC 取数据中最频繁动作')

## ✏️ 练习 2：分布偏移度量

实现 `ood_action_fraction(policy_greedy, data)`：给定一个贪婪策略（每状态一个动作），返回它选了**数据未覆盖动作**的状态比例（衡量策略偏离数据支撑的程度）。

In [ ]:
def ood_action_fraction(policy_greedy, data):
    # TODO: 统计 (s, policy_greedy[s]) 在数据中计数为 0 的状态比例
    #       (只数据里出现过的状态 s)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
data = collect_dataset()
# 一个故意在 s=2 选 OOD(左) 的策略
bad = np.ones(N, dtype=int); bad[2] = 0          # s=2 选左(OOD!)
good = np.ones(N, dtype=int)                     # 全右(全在数据内)
frac_bad = ood_action_fraction(bad, data)
frac_good = ood_action_fraction(good, data)
assert frac_bad > 0, '在 s=2 选 OOD 的策略应有 >0 的 OOD 比例'
assert frac_good == 0.0, '全右策略全在数据内 -> OOD 比例 0'
print(f'坏策略 OOD 比例={frac_bad:.2f}, 好策略 OOD 比例={frac_good:.2f}')
print('✅ 练习 2 通过：能度量策略偏离数据支撑的程度')

## ✏️ 练习 3：CQL 保守惩罚梯度

实现 `cql_penalty_grad(Q_s, a_data)`：给定某状态的 Q 向量 `Q_s`（长度 nA）与数据动作 `a_data`，返回保守项对 `Q_s` 的梯度 = `softmax(Q_s) - onehot(a_data)`（压所有、抬数据内）。

In [ ]:
def cql_penalty_grad(Q_s, a_data):
    # TODO: soft = softmax(Q_s); grad = soft - onehot(a_data)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
Q_s = np.array([5.0, 1.0])           # 动作0(OOD)被高估
g = cql_penalty_grad(Q_s, a_data=1)  # 数据动作是 1
# 梯度下降会: 压低高 Q 的动作0(soft 大)、抬高数据动作1
assert g[0] > 0, 'OOD 高估动作的梯度>0 -> 被压低'
assert g[1] < 0, '数据动作的梯度<0 -> 被抬高'
assert abs(g.sum()) < 1e-9, 'softmax 和为1，减 onehot 后梯度和为0'
print('CQL 惩罚梯度:', np.round(g, 3), '(动作0被压、动作1被抬)')
print('✅ 练习 3 通过：CQL 保守惩罚压 OOD、抬数据内')

## ✏️ 练习 4：expectile 回归

实现 `expectile_value(values, tau, iters, lr)`：用梯度下降把一个标量拟合到 `values` 的 τ-expectile。
(梯度 = `-2·w·(target-pred)`，`w = τ` if `target>pred` else `1-τ`。)

In [ ]:
def expectile_value(values, tau, iters=3000, lr=0.01):
    # TODO: v=0.0; 迭代: diff=values-v; w=where(diff>0,tau,1-tau); v -= lr*mean(-2*w*diff)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
vals = np.array([0.0, 1.0, 2.0, 10.0])
v_mean = expectile_value(vals, 0.5)
v_high = expectile_value(vals, 0.95)
assert abs(v_mean - vals.mean()) < 0.1, 'tau=0.5 -> 均值'
assert v_high > v_mean + 1.0, 'tau=0.95 -> 明显偏向 max'
assert v_high < vals.max(), 'expectile 不会超过 max'
print(f'tau=0.5: {v_mean:.2f} (均值={vals.mean():.1f}), tau=0.95: {v_high:.2f} (max={vals.max():.0f})')
print('✅ 练习 4 通过：expectile 用 tau 在均值与 max 间滑动')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1
def bc_greedy_policy(data):
    counts = np.zeros((N, nA))
    for s, a, *_ in data: counts[int(s), int(a)] += 1
    return counts.argmax(axis=1)

In [ ]:
# 练习 2
def ood_action_fraction(policy_greedy, data):
    cov = np.zeros((N, nA)); seen = set()
    for s, a, *_ in data: cov[int(s), int(a)] += 1; seen.add(int(s))
    ood = sum(1 for s in seen if cov[s, policy_greedy[s]] == 0)
    return ood / len(seen)

In [ ]:
# 练习 3
def cql_penalty_grad(Q_s, a_data):
    soft = np.exp(Q_s - Q_s.max()); soft /= soft.sum()
    onehot = np.zeros_like(Q_s); onehot[a_data] = 1.0
    return soft - onehot

In [ ]:
# 练习 4
def expectile_value(values, tau, iters=3000, lr=0.01):
    v = 0.0
    for _ in range(iters):
        diff = values - v
        w = np.where(diff > 0, tau, 1 - tau)
        v -= lr * np.mean(-2 * w * diff)
    return v

---
## 🧪 真实数据胶囊：D4RL 基准与离线分数

**D4RL**（Fu 2020）是离线 RL 的标准基准。同一算法在不同**数据质量**上分数天差地别——这正是「分布偏移之难在于分布而非数量」的实证。下面是一组**代表性的归一化分数**（约数，来自公开论文）。

In [ ]:
# D4RL MuJoCo 归一化分数（约数；100=专家, 0=随机）。展示数据质量的决定性影响。
D4RL_SCORES = {
    # 数据集质量      BC     CQL    IQL
    'halfcheetah-medium':     dict(BC=42.6, CQL=44.0, IQL=47.4),
    'halfcheetah-medium-replay': dict(BC=36.6, CQL=45.5, IQL=44.2),
    'hopper-medium':          dict(BC=52.9, CQL=58.5, IQL=66.3),
    'hopper-medium-expert':   dict(BC=52.5, CQL=98.7, IQL=91.5),
}
print(f"{'数据集':28s}{'BC':>7}{'CQL':>7}{'IQL':>7}")
for name, sc in D4RL_SCORES.items():
    print(f'{name:28s}{sc["BC"]:>7.1f}{sc["CQL"]:>7.1f}{sc["IQL"]:>7.1f}')
# 观察1：值方法(CQL/IQL)在混合/medium-replay 上明显超过 BC(拼接能力)
assert D4RL_SCORES['halfcheetah-medium-replay']['CQL'] > D4RL_SCORES['halfcheetah-medium-replay']['BC']
# 观察2：medium-expert(含专家数据)上所有方法都高，BC 也不差
assert D4RL_SCORES['hopper-medium-expert']['CQL'] > 90
print('\n✅ 值方法(CQL/IQL)在次优/混合数据上靠拼接超过 BC；专家数据上差距缩小')

**🧪 胶囊练习**：实现 `improvement_over_bc(scores)`：给定一个数据集的 `dict(BC=, CQL=, IQL=)`，返回 CQL 与 IQL 相对 BC 的分数提升 `(cql_gain, iql_gain)`。

In [ ]:
def improvement_over_bc(scores):
    # TODO: 返回 (scores['CQL']-scores['BC'], scores['IQL']-scores['BC'])
    raise NotImplementedError

In [ ]:
# 自测
g = improvement_over_bc(D4RL_SCORES['hopper-medium'])
assert abs(g[0] - (58.5-52.9)) < 1e-6 and abs(g[1] - (66.3-52.9)) < 1e-6
print(f'hopper-medium: CQL 比 BC +{g[0]:.1f}, IQL 比 BC +{g[1]:.1f}')
print('✅ 胶囊练习通过：值方法在次优数据上的拼接增益')

In [ ]:
# 📖 胶囊参考答案
def improvement_over_bc(scores):
    return scores['CQL'] - scores['BC'], scores['IQL'] - scores['BC']

### 小结
- **离线 RL**：只给固定数据集、不许再交互。死结 = **分布偏移 + 外推误差**：OOD 动作被 `max` 高估、却因不交互而**永不被纠正**，自举把幻觉越滚越大。
- **BC**：直接模仿数据，安全(不碰 OOD)但天花板=数据水平、不会拼接。
- **CQL**：TD 损失 + 保守正则(压 logsumexp、抬数据内动作)，学真实 Q 的**下界**，把 OOD 的 Q 压到数据内之下。
- **IQL**：用高-τ **expectile 回归**在**只看数据内动作**时近似 max、绕开 OOD 查询；再用优势加权回归(AWR)抽数据内策略。
- **统一视角**：在线**乐观**探索、离线**悲观**保守——因为离线不能用试错纠错。

你已亲手看到离线死结、并实现三种解药。下一站：**模块 04 · 基于模型与世界模型** —— 换个思路省交互：学一个模型来代替真实环境。